In [26]:
import numpy as np

import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
device

'mps'

In [27]:
import tiktoken
import torch as t
from torch.nn import Embedding
from torch.nn.functional import cross_entropy

testing_mode = True
encoder = tiktoken.get_encoding("cl100k_base")

In [28]:

from datasets import load_dataset

ds = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)


In [29]:
for i, doc in enumerate(ds):
    print(doc["text"][:100], "\n\n", len(doc["text"]))
    print(len(encoder.encode(doc["text"][:100])))
    if i==10:
        break

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are rea 

 3665
24
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night 

 5255
30
How do you get HIV?
HIV can be passed on when infected bodily fluid, such as blood or semen, is pass 

 628
25
CTComms sends on average 2 million emails monthly on behalf of over 125 different charities and not  

 17410
22
Hold the salt: UCLA engineers develop revolutionary new desalination membrane
Process uses atmospher 

 5785
18
Not Just for Kids
The Hunt for Falling Leaves...
Nature's Color on the Ground
by Mary Catherine Ball 

 3204
22
The Solar and Heliospheric Observatory (SOHO) spacecraft is expected to discover its 1,000TH comet t 

 2909
24
Bolivia: Coca-chewing protest outside US embassy
Indigenous activists in Bolivia have been holding a 

 2246
21
Breaking the COX code
Using the team approach
Editor’s Note: This story, first published in 2004, ha 

 

In [30]:
block_size = 128
context_length = 512
stride = int(block_size / 2)
input_tokens = []
output_tokens = []
for i, doc in enumerate(ds):
    if doc["language_score"] < 0.7 or doc["score"] < 3:
        continue
    tokens = encoder.encode(doc["text"])
    # adding the end-of-text token to the end
    tokens.append(encoder.eot_token)
    len_tokens = len(tokens)
    for i in range(0, len_tokens, stride):  # tryin to reach a sweet spot between overlapping and non overlapping
        start1 = i
        end1 = block_size + i
        if end1 + block_size > len_tokens:
            break
        input_tokens.append(tokens[start1: end1])
        output_tokens.append(tokens[start1+1: end1 + 1])
    if testing_mode:
        if len(input_tokens) > 10000:  # for testing
            break
# Convert once (VERY IMPORTANT)
input_tokens = torch.tensor(input_tokens).to(device)
output_tokens = torch.tensor(output_tokens).to(device)
# print(input_tokens, output_tokens)
# print(len())

In [31]:
assert input_tokens.shape == output_tokens.shape

In [32]:
tensor_ds = t.utils.data.TensorDataset(input_tokens, output_tokens)

In [33]:
class SingleHeadCausalAttention(torch.nn.Module):
    def __init__(self, head_dim: int, embed_dim=384, drop_out: float = 0.2):
        super().__init__()
        self.head_dim = head_dim
        self.embed_dim = embed_dim
        self.Wq = t.nn.Linear(in_features=embed_dim, out_features=head_dim, device=device)
        self.Wk = t.nn.Linear(in_features=embed_dim, out_features=head_dim, device=device)
        self.Wv = t.nn.Linear(in_features=embed_dim, out_features=head_dim, device=device)
        self.drop_out = t.nn.Dropout(drop_out)

    def forward(self, inputs):
        query_t = self.Wq(inputs)
        key_t = self.Wk(inputs)
        val_t = self.Wv(inputs)

        attention_score = query_t @ key_t.transpose(1, 2)

        # here we need to make upper triangular matrix as -inf
        T = attention_score.shape[-1]
        mask = t.triu(t.ones(T, T, device=inputs.device), diagonal=1).bool()
        attention_weights = attention_score.masked_fill(mask, float('-inf'))

        att_weights = t.softmax(attention_weights / self.head_dim ** 0.5, dim=-1)
        att_weights_dropped = self.drop_out(att_weights)
        return att_weights_dropped @ val_t

In [34]:
class MultiHeadAttention(t.nn.Module):
    def __init__(self, embed_dim=384, no_of_att_heads=6, drop_out:float=0.2):
        super().__init__()
        dim_per_head = embed_dim // no_of_att_heads
        self.modulesList = t.nn.ModuleList(
            [SingleHeadCausalAttention(head_dim=dim_per_head, embed_dim=embed_dim) for _ in range(no_of_att_heads)])
        self.out_proj = t.nn.Linear(in_features=embed_dim,
                                    out_features=embed_dim)  #mixes the head outputs to get most useful features out

    def forward(self, inputs):
        multi_head_out = [head(inputs) for head in self.modulesList]
        concat_out = t.cat(multi_head_out, dim=-1)
        return self.out_proj(concat_out)

In [35]:
class TransformerBlock(t.nn.Module):
    def __init__(self, no_of_heads: int, embed_dim, drop_out = 0.2):
        super().__init__()
        self.embed_dim = embed_dim
        #first is layer normalization
        self.layer_n_before = t.nn.LayerNorm(embed_dim)
        self.drop_out1 = t.nn.Dropout(drop_out)
        self.drop_out2 = t.nn.Dropout(drop_out)
        self.layer_n_after = t.nn.LayerNorm(embed_dim)
        self.multi_head_att = MultiHeadAttention(embed_dim, no_of_heads)
        self.feed_for_netw = t.nn.Sequential(t.nn.Linear(embed_dim, 5 * embed_dim), t.nn.GELU(),
                                             t.nn.Linear(5 * embed_dim, embed_dim))

    def forward(self, inputs):
        context_vector = self.drop_out1(self.multi_head_att(self.layer_n_before(inputs)))
        x = context_vector + inputs

        norm = self.layer_n_after(x)
        ffn_output = self.feed_for_netw(norm)
        ffn_output = self.drop_out2(ffn_output)
        return ffn_output + x


In [36]:
import math

class SinusoidalPositionalEmbedding(t.nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = t.zeros(max_len, d_model)
        position = t.arange(0, max_len, dtype=t.float).unsqueeze(1)
        div_term = t.exp(t.arange(0, d_model, 2, dtype=t.float) * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = t.sin(position * div_term)
        pe[:, 1::2] = t.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, seq_len: int) -> t.Tensor:
        return self.pe[:seq_len, :]

class GPTStyleTransformer(t.nn.Module):
    def __init__(self, no_of_heads, embed_dim, no_of_trans_blocks, vocab_size, drop_out=0.2):
        super().__init__()
        self.embed_dim = embed_dim
        self.token_embed = t.nn.Embedding(vocab_size, embedding_dim=embed_dim)
        self.pos_embed = SinusoidalPositionalEmbedding(d_model=embed_dim)
        self.embed_dropout = t.nn.Dropout(drop_out)
        self.transformers = t.nn.ModuleList(
            [TransformerBlock(no_of_heads, embed_dim) for _ in range(no_of_trans_blocks)])
        self.layer_norm = t.nn.LayerNorm(embed_dim)
        self.lm_head = t.nn.Linear(embed_dim, vocab_size)
        self.lm_head.weight = self.token_embed.weight

    def forward(self, input_token_ids):  # here the inputs will be token embeddings + positional Embeddings
        token_embed = self.token_embed(input_token_ids)
        seq_len = input_token_ids.shape[-1]
        pos_embed = self.pos_embed(seq_len)
        add_embeddings = self.embed_dropout(token_embed + pos_embed)
        data = add_embeddings
        for index, trans in enumerate(self.transformers):
            data = trans(data)

        return self.lm_head(self.layer_norm(data))


In [37]:
trans = GPTStyleTransformer(no_of_heads=6, embed_dim=384, no_of_trans_blocks=6, vocab_size=100277, drop_out=0.1).to(device)
model = trans

In [38]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {trainable_params:,}")

# Assuming you've instantiated the model like so:
# model = GPTStyleTransformer(...)

print_trainable_parameters(trans)


Total Trainable Parameters: 51,025,973


In [39]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
model.train()

GPTStyleTransformer(
  (token_embed): Embedding(100277, 384)
  (pos_embed): SinusoidalPositionalEmbedding()
  (embed_dropout): Dropout(p=0.1, inplace=False)
  (transformers): ModuleList(
    (0-5): 6 x TransformerBlock(
      (layer_n_before): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (drop_out1): Dropout(p=0.2, inplace=False)
      (drop_out2): Dropout(p=0.2, inplace=False)
      (layer_n_after): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (multi_head_att): MultiHeadAttention(
        (modulesList): ModuleList(
          (0-5): 6 x SingleHeadCausalAttention(
            (Wq): Linear(in_features=384, out_features=64, bias=True)
            (Wk): Linear(in_features=384, out_features=64, bias=True)
            (Wv): Linear(in_features=384, out_features=64, bias=True)
            (drop_out): Dropout(p=0.2, inplace=False)
          )
        )
        (out_proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (feed_for_netw): Sequential

In [41]:
from torch.utils.data import DataLoader
# Wrap your dataset in a DataLoader to add the Batch dimension!
train_loader = DataLoader(tensor_ds, batch_size=32, shuffle=True)
for step, (x_batch, y_batch) in enumerate(train_loader):
    logits = model(x_batch)
    loss = t.nn.functional.cross_entropy(
            logits.view(-1, encoder.n_vocab), 
            y_batch.view(-1)
        )
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
     # Logging
    if step % 10 == 0:
        print(f"Step {step:07d} | Loss: {loss.item():.4f}")

Step 0000000 | Loss: 272.7934
Step 0000010 | Loss: 62.1820
Step 0000020 | Loss: 53.3218
Step 0000030 | Loss: 49.0548
Step 0000040 | Loss: 45.2182
Step 0000050 | Loss: 43.3226
Step 0000060 | Loss: 40.2218
Step 0000070 | Loss: 38.9562
Step 0000080 | Loss: 37.9761
Step 0000090 | Loss: 36.1646
Step 0000100 | Loss: 34.5277
Step 0000110 | Loss: 33.1549
Step 0000120 | Loss: 31.7692
Step 0000130 | Loss: 30.2403
Step 0000140 | Loss: 29.2675
Step 0000150 | Loss: 29.3984
Step 0000160 | Loss: 27.5895
Step 0000170 | Loss: 25.6665
Step 0000180 | Loss: 25.1722
Step 0000190 | Loss: 24.4016
Step 0000200 | Loss: 24.0513
Step 0000210 | Loss: 23.4940
Step 0000220 | Loss: 22.3685
Step 0000230 | Loss: 22.6297
Step 0000240 | Loss: 21.6274
Step 0000250 | Loss: 20.8203
Step 0000260 | Loss: 20.3131
Step 0000270 | Loss: 19.4545
Step 0000280 | Loss: 19.8412
Step 0000290 | Loss: 18.5853
Step 0000300 | Loss: 18.1922
Step 0000310 | Loss: 17.6516


In [43]:
model.eval()
question = "What is the capital of "

# Encode the prompt and push it to the correct device
context = torch.tensor(encoder.encode(question), dtype=torch.long).unsqueeze(0).to(device)

# Turn off gradients for pure generation speed
with torch.no_grad():
    for _ in range(50): # generate up to 50 new tokens
        # Ensure context doesn't exceed our 128 block_size limit!
        idx_cond = context[:, -block_size:]
        
        # Forward pass
        logits = model(idx_cond)
        
        # Focus strictly on the last time step emitted
        logits = logits[:, -1, :] 
        
        # Apply Softmax to map logits into probabilities
        probs = t.nn.functional.softmax(logits, dim=-1)
        
        # Sample the next token from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)
        
        # Append the new token back into the sequence context
        context = torch.cat((context, idx_next), dim=1)
        
        # The model mathematically terminates its own thought
        if idx_next.item() == encoder.eot_token:
            break

# Print out the final aggregated text sequence
print(encoder.decode(context[0].tolist()))



What is the capital of  In the can, of the be like like like,ateral of the illegal, and theannah of the architectural the Support, of the cell cell is to the identifies of the towns"" of bass, in the for for pressure to the on on
